<a href="https://colab.research.google.com/github/Vaibhav-Magadum/Blockchain_Assignments/blob/main/Week5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import time
import yaml
from pathlib import Path
from typing import Iterable

In [2]:
from docling_core.types.doc import ImageRefMode
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.document import ConversionResult
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption


In [3]:
USE_V2 = True
USE_LEGACY = False

In [4]:
def export_documents_markdown_only(
    conv_results: Iterable[ConversionResult],
    output_dir: Path,
):
    output_dir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    failure_count = 0
    partial_success_count = 0

    for conv_res in conv_results:
        if conv_res.status == ConversionStatus.SUCCESS:
            success_count += 1
            doc_filename = conv_res.input.file.stem

            if USE_V2:
                out_file = output_dir / doc_filename
                out_file.parent.mkdir(parents=True, exist_ok=True)
                with (out_file.with_suffix(".md")).open("w") as fp:
                    fp.write(conv_res.document.export_to_markdown())

            if USE_LEGACY:
                out_file = output_dir / doc_filename
                out_file.parent.mkdir(parents=True, exist_ok=True)
                with (out_file.with_suffix(".legacy.md")).open("w", encoding="utf-8") as fp:
                    fp.write(conv_res.legacy_document.export_to_markdown())

        elif conv_res.status == ConversionStatus.PARTIAL_SUCCESS:
            partial_success_count += 1
        else:
            failure_count += 1

    return success_count, partial_success_count, failure_count


In [5]:
def main():
    input_doc_paths = [
        Path("/content/Automation_in_Indian_Mining_Industries.pdf"),
        Path("/content/MACHINERY AND EQUIPMENT FOR MINING.pdf"),
        Path("/content/Mining Checklist.pdf"),
        Path("/content/Product Catalog for mining products.pdf"),
        Path("/content/SAFETY IN COAL MINES.pdf"),
        Path("/content/Safety & health in small-scale surface.pdf"),
    ]

    pipeline_options = PdfPipelineOptions()
    pipeline_options.generate_page_images = True

    doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options, backend=DoclingParseV4DocumentBackend
            )
        }
    )

    start_time = time.time()

    conv_results = doc_converter.convert_all(
        input_doc_paths,
        raises_on_error=False,
    )
    success_count, partial_success_count, failure_count = export_documents_markdown_only(
        conv_results, output_dir=Path("scratch")
    )

    end_time = time.time() - start_time

    print(f"Document conversion complete in {end_time:.2f} seconds.")
    print(
        f"Success: {success_count}, Partial: {partial_success_count}, Failed: {failure_count}"
    )

    if failure_count > 0:
        raise RuntimeError(
            f"The example failed converting {failure_count} on {len(input_doc_paths)}."
        )


if __name__ == "__main__":
    main()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Document conversion complete in 423.39 seconds.
Success: 6, Partial: 0, Failed: 0


In [6]:
import time
from pathlib import Path
from docling_core.types.doc import ImageRefMode
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.document import TableItem, PictureItem

In [7]:
IMAGE_RESOLUTION_SCALE = 2.0

In [8]:
def process_pdf_with_images(input_pdf_path: Path, output_dir: Path):
    # Setup pipeline options to keep and scale images
    pipeline_options = PdfPipelineOptions()
    pipeline_options.images_scale = IMAGE_RESOLUTION_SCALE
    pipeline_options.generate_page_images = True
    pipeline_options.generate_picture_images = True

    # Initialize document converter for PDF format with pipeline options
    doc_converter = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
    )

    start_time = time.time()
    conv_res = doc_converter.convert(input_pdf_path)

    output_dir.mkdir(parents=True, exist_ok=True)
    doc_filename = conv_res.input.file.stem

    # Save page images as PNG
    for page_no, page in conv_res.document.pages.items():
        page_image_filename = output_dir / f"{doc_filename}-{page.page_no}.png"
        with page_image_filename.open("wb") as fp:
            page.image.pil_image.save(fp, format="PNG")

    # Save images of tables and pictures
    table_counter = 0
    picture_counter = 0
    for element, _level in conv_res.document.iterate_items():
        if isinstance(element, TableItem):
            table_counter += 1
            element_image_filename = output_dir / f"{doc_filename}-table-{table_counter}.png"
            with element_image_filename.open("wb") as fp:
                element.get_image(conv_res.document).save(fp, "PNG")

        elif isinstance(element, PictureItem):
            picture_counter += 1
            element_image_filename = output_dir / f"{doc_filename}-picture-{picture_counter}.png"
            with element_image_filename.open("wb") as fp:
                element.get_image(conv_res.document).save(fp, "PNG")

    # Save markdown with embedded pictures
    md_embedded_filename = output_dir / f"{doc_filename}-with-images.md"
    conv_res.document.save_as_markdown(md_embedded_filename, image_mode=ImageRefMode.EMBEDDED)

    # Save markdown with externally referenced pictures
    md_ref_filename = output_dir / f"{doc_filename}-with-image-refs.md"
    conv_res.document.save_as_markdown(md_ref_filename, image_mode=ImageRefMode.REFERENCED)

    # Save HTML with externally referenced pictures
    html_filename = output_dir / f"{doc_filename}-with-image-refs.html"
    conv_res.document.save_as_html(html_filename, image_mode=ImageRefMode.REFERENCED)

    elapsed = time.time() - start_time
    print(f"Document converted and images exported in {elapsed:.2f} seconds.")



In [9]:
def main():
    input_doc_path = Path("/content/Automation_in_Indian_Mining_Industries.pdf")
    output_dir = Path("scratch")

    process_pdf_with_images(input_doc_path, output_dir)


if __name__ == "__main__":
    main()

Document converted and images exported in 266.51 seconds.


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os
import json
from pathlib import Path

In [6]:
markdown_dir = Path("/content/scratch")
output_jsonl = "/content/generated_qa.jsonl"

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from markdownify import markdownify as md
import os, glob, json
from tqdm import tqdm

In [2]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    load_in_4bit=True,
    trust_remote_code=True
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [3]:
qa_generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

Device set to use cuda:0


In [4]:
def chunk_text(text, max_tokens=800):
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
    words = text.split("\n\n")
    chunks = []
    current = ""
    for para in words:
        if len(enc.encode(current + para)) < max_tokens:
            current += para + "\n\n"
        else:
            chunks.append(current.strip())
            current = para + "\n\n"
    if current:
        chunks.append(current.strip())
    return chunks

In [ ]:
input_dir = "/content/scratch"
output_file = "/content/generated_qa_dataset.jsonl"

with open(output_file, "w") as outfile:
    for md_file in tqdm(glob.glob(os.path.join(input_dir, "*.md"))):
        with open(md_file, "r") as f:
            raw_md = f.read()

        plain_text = md(raw_md)
        chunks = chunk_text(plain_text)

        for chunk in chunks:
            prompt = f"""### Instruction:
Generate a set of question-answer pairs from the following technical text.

### Input:
{chunk}

### Response:
"""
            output = qa_generator(prompt, max_new_tokens=512, do_sample=True, temperature=0.7)[0]["generated_text"]

            # Extract only the Q&A part
            qa_output = output.split("### Response:")[-1].strip()
            entry = {
                "source_file": os.path.basename(md_file),
                "chunk": chunk,
                "qa_pairs": qa_output
            }
            outfile.write(json.dumps(entry) + "\n")

print(f"\n✅ Done. Output saved to: {output_file}")

  0%|          | 0/8 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.11/dist-packages/bitsandbytes/nn/modules.py:451: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
 12%|█▎        | 1/8 [05:42<39:57, 342.47s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-en

In [2]:
!pip install -q langchain langchain-core
!pip install -q --upgrade transformers accelerate

In [1]:
from transformers import pipeline

# Load evaluator model (lightweight)
evaluator = pipeline("text2text-generation", model="google/flan-t5-base")

# Function to evaluate a Q&A pair
def evaluate_qa(chunk, qa_text):
    prompt = f"""Evaluate the following question-answer pairs for clarity, relevance, and completeness with respect to the input technical text.

### Text:
{chunk}

### Q&A:
{qa_text}

### Evaluation:
Rate the Q&A on:
- Relevance (0-5)
- Clarity (0-5)
- Completeness (0-5)

Respond in this format:
Relevance: X
Clarity: Y
Completeness: Z
"""
    result = evaluator(prompt, max_new_tokens=100)[0]['generated_text']
    return result

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cuda:0


In [2]:
import json
from tqdm import tqdm

qa_file = "/content/generated_qa_dataset.jsonl"
eval_file = "/content/evaluated_qa_dataset.jsonl"

with open(qa_file, "r") as infile, open(eval_file, "w") as outfile:
    for line in tqdm(infile):
        entry = json.loads(line)
        eval_result = evaluate_qa(entry["chunk"], entry["qa_pairs"])
        entry["evaluation"] = eval_result
        outfile.write(json.dumps(entry) + "\n")

print(f"✅ Evaluation done. Results saved to {eval_file}")

0it [00:00, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (664 > 512). Running this sequence through the model will result in indexing errors
10it [00:07,  2.04it/s]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
30it [00:20,  1.48it/s]

✅ Evaluation done. Results saved to /content/evaluated_qa_dataset.jsonl


In [ ]:
from transformers import pipeline
from tqdm import tqdm
import json

# Load pipeline with Mistral
qa_generator = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.2",
    device=0
)

def regenerate_qa(chunk):
    prompt = f"""Generate 3 technical question-answer pairs based on the following technical content:

{chunk}

Format:
Q1: ...
A1: ...
Q2: ...
A2: ...
Q3: ...
A3: ...
"""

    output = qa_generator(
        prompt,
        max_new_tokens=1024,  # increase token limit
        do_sample=False,
        eos_token_id=qa_generator.tokenizer.eos_token_id
    )

    return output[0]["generated_text"]

# Load your bad entries (assuming you already have them)
regenerated_entries = []

for entry in tqdm(bad_entries):
    chunk = entry["chunk"]
    try:
        new_qa = regenerate_qa(chunk)
        entry["qa_pairs"] = new_qa
        regenerated_entries.append(entry)
    except Exception as e:
        print(f"❌ Error on chunk: {e}")

# Save to JSONL
with open("/content/regenerated_qa_dataset.jsonl", "w") as outfile:
    for entry in regenerated_entries:
        outfile.write(json.dumps(entry) + "\n")

print("✅ Regeneration complete.")


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
import random
import json

# Load final dataset
with open("/content/generated_qa_dataset.jsonl", "r") as f:
    entries = [json.loads(line) for line in f]

# Pick a random entry
sample_entry = random.choice(entries)
print("📄 Context chunk:\n", sample_entry["chunk"][:500])  # Truncated
print("\n❓ Available Q&A:\n", sample_entry["qa_pairs"])


📄 Context chunk:
 ## Photo F-1
- ∙ Cylinder roller bearing inner ring
- ∙ alse brinelling has occurred around the total circumference of the raceway F surface
- ∙ Caused by vibration
## Photo F-3
- ∙ Cylinder roller bearing outer ring
- ∙ Fretting corrosionn has occurred along the outer diameter
## Photo F-2
- ∙ Deep groove ball bearing inner ring
- ∙ alse brinelling has occurred around the total circumference of the raceway F surface
- ∙ Caused by vibration
## Photo F-4
- ∙ Tapered roller bearing outer ring
- ∙ 

❓ Available Q&A:
 **Question:** What parts in Photo F-1 and F-2 show signs of subsurface damage?
**Answer:** The inner rings of the cylinder roller bearings in both Photo F-1 and F-2 exhibit subsurface damage, specifically alse brinelling around the entire circumference of the raceway F surface, which is caused by vibration.

**Question:** What type of bearing is shown in Photo F-3 and F-4, and where has corrosion occurred?
**Answer:** The bearings in Photo F-3 and F-4 are cy